# Data view — Kaggle raw vs. pilkwang labels

Two tables, side by side, for the same studies:

| source | file | what it holds |
|---|---|---|
| **Kaggle (competition)** | `data/train.csv` | one row per study: the free-text `Report` plus 12 label columns — but only **58** studies are actually labelled, the rest are blank |
| **pilkwang** | `data/meta/rsna-knee-llm-labels-pilkwang/report_labels_v2.csv` | one row per study: the same 12 findings, **every** study filled in, each as a `score` / `__conf` / `__verdict` triplet an LLM read out of the report |

They key on the same `StudyInstanceUID`, so the two can be lined up row for row.

One correction to the sketch this notebook grew from: **pilkwang carries no report text at all** — it is
labels only. The reports live only in the Kaggle CSV and are multi-language (Spanish, Turkish, Greek,
Croatian, German, Bulgarian, Dutch, French, English…). What is "in English" on the pilkwang side is the
*label vocabulary* (`YES` / `NO` / `UNK`), produced by reading each report in its original language.
Section 3 checks that claim against the data instead of assuming it.

If the files are missing, run [00-kaggleFetch.ipynb](../00-kaggleFetch.ipynb) first (Kaggle CSVs), and
pull the label set with `kaggle datasets download -d pilkwang/rsna-knee-llm-labels`.

In [1]:
from pathlib import Path
import re
import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

# Repo root, never the cwd - this notebook sits two levels deep in notebooks/niko/.
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data"

KAGGLE_CSV = DATA / "train.csv"
PILKWANG_CSV = DATA / "meta" / "rsna-knee-llm-labels-pilkwang" / "report_labels_v2.csv"

# The 12 findings, in competition order. Order is load-bearing downstream.
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

N_SAMPLE = 100   # rows to render
SEED = 42        # fixed, so the same 100 studies come back on a re-run

for label, path, how in [
    ("Kaggle train.csv", KAGGLE_CSV, "run notebooks/00-kaggleFetch.ipynb"),
    ("pilkwang labels", PILKWANG_CSV, "kaggle datasets download -d pilkwang/rsna-knee-llm-labels"),
]:
    print("{:<18} {:<7} {}".format(label, "OK" if path.exists() else "MISSING",
                                   path.relative_to(ROOT) if path.exists() else how))

Kaggle train.csv   OK      data/train.csv
pilkwang labels    OK      data/meta/rsna-knee-llm-labels-pilkwang/report_labels_v2.csv


## 1 — Kaggle raw

`train.csv` as it ships: the keys (columns), and 100 random studies as rows.

In [2]:
# Get the raw data from kaggle and render here the keys and values (random 100)

kaggle = pd.read_csv(KAGGLE_CSV)
print("data/train.csv ->", kaggle.shape[0], "rows x", kaggle.shape[1], "columns")
print("unique StudyInstanceUID:", kaggle["StudyInstanceUID"].nunique())

# --- the keys, and how much of each one is actually filled in ---
keys_kaggle = pd.DataFrame({
    "dtype": kaggle.dtypes.astype(str),
    "non_null": kaggle.notna().sum(),
    "filled_%": (100 * kaggle.notna().mean()).round(1),
    "example": [str(kaggle[c].dropna().iloc[0])[:60] if kaggle[c].notna().any() else "-"
                for c in kaggle.columns],
})
display(keys_kaggle)

data/train.csv -> 4407 rows x 14 columns
unique StudyInstanceUID: 4407


,dtype,non_null,filled_%,example
StudyInstanceUID,object,4407,100.0,1.2.826.0.1.3680043.8.498.1000487322909905386909332429219581
Report,object,4407,100.0,Técnica: RMN de la rodilla. Resultados: Rotura de menisco in
ACL,float64,58,1.3,0.0
MCL,float64,58,1.3,0.0
Medial Meniscus,float64,58,1.3,0.0
Lateral Meniscus,float64,58,1.3,0.0
Medial OA,float64,58,1.3,0.0
Lateral OA,float64,58,1.3,0.0
PF OA,float64,58,1.3,1.0
Effusion,float64,58,1.3,1.0


In [3]:
# --- 100 random studies (seeded, so re-running gives the same 100) ---
sample_uids = kaggle["StudyInstanceUID"].sample(n=N_SAMPLE, random_state=SEED).tolist()

kaggle_idx = kaggle.set_index("StudyInstanceUID")
kaggle_sample = kaggle_idx.loc[sample_uids].copy()
# Reports carry embedded newlines - flatten them or the table is unreadable.
kaggle_sample["Report"] = (kaggle_sample["Report"].astype(str)
                           .str.replace(r"\s+", " ", regex=True).str.slice(0, 120) + "...")

print("Kaggle - {} random studies. Label columns are blank for all but the 58 gold studies.".format(N_SAMPLE))
display(kaggle_sample)

Kaggle - 100 random studies. Label columns are blank for all but the 58 gold studies.


,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
StudyInstanceUID,,,,,,,,,,,,,
1.2.826.0.1.3680043.8.498.83376388091077619281568190257019735638,Regelrechte Stellungsverhältnisse im Kniegelenk. Allseits regelrechtes Knochenmarksign...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1.2.826.0.1.3680043.8.498.11938864543152001817866013174907274014,"SAĞ DİZ MRG. Tetkik protokolü: Çok düzlemli, çok sekanslı. Bulgular: Medial meniscüs p...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1.2.826.0.1.3680043.8.498.49394404966674633152360206835416205330,Prikazane koštane strukture primjerenih intenziteta signala i morfologije. Prikazana z...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1.2.826.0.1.3680043.8.498.11402325536521359105285323485008939602,ΤΕΧΝΙΚΗ: Η εξέταση έγινε σε Τοµογράφο 3Τ µε ακολουθίες παλµών pd/T2 fsκαι Τ1. ΕΥΡΗΜΑΤΑ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1.2.826.0.1.3680043.8.498.11545541207342029419461179316053148974,"MRI of right knee in sagittal section (Proton density with fat saturation, T1WI, T2WI ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1.2.826.0.1.3680043.8.498.19512754570664577075249836633718382506,"МР находка: Костите, формиращи дясната колянна става са с нормална форма. Артикулиращи...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1.2.826.0.1.3680043.8.498.43445351322695981909807931874796758490,Diz eklemi içi sıvı miktarı normal. Kuadriseps yağ yastıkcığındaki hafif dereceli ödem...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1.2.826.0.1.3680043.8.498.91860013313252950044036077977399321593,Interprétation : Minime liséré de liquide au sein du récessus supra-patellaire sans ép...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# How many studies carry real labels at all?
gold_mask = kaggle[LABELS].notna().all(axis=1)
print("fully labelled studies :", int(gold_mask.sum()), "/", len(kaggle))
print("partially labelled     :", int((kaggle[LABELS].notna().any(axis=1) & ~gold_mask).sum()))
print("report but no labels   :", int((~kaggle[LABELS].notna().any(axis=1)).sum()))
n_gold_in_sample = int(kaggle_idx.loc[sample_uids, LABELS].notna().all(axis=1).sum())
print("\n{} of the {} sampled studies {} labels".format(
    n_gold_in_sample, N_SAMPLE, "carries" if n_gold_in_sample == 1 else "carry"))

fully labelled studies : 58 / 4407
partially labelled     : 0
report but no labels   : 4349

1 of the 100 sampled studies carries labels


## 2 — pilkwang

Same studies, same 12 findings, but every row filled in. Each finding arrives as three columns:

- `<finding>` — a soft score in `[0, 1]`
- `<finding>__conf` — how confident the reader was
- `<finding>__verdict` — `YES` / `NO` / `UNK`, where `UNK` means *the report never mentions this finding*

In [5]:
# Get the same data from pilkwang and render keys and values (random 100)

pilkwang = pd.read_csv(PILKWANG_CSV)
print("report_labels_v2.csv ->", pilkwang.shape[0], "rows x", pilkwang.shape[1], "columns")
print("unique StudyInstanceUID:", pilkwang["StudyInstanceUID"].nunique())
print("has a report text column:", any("report" in c.lower() for c in pilkwang.columns))

# --- the keys: 12 findings x (score, conf, verdict) ---
keys_pilkwang = pd.DataFrame([
    {"finding": l,
     "score_col": l, "score_range": "{:.2f} - {:.2f}".format(pilkwang[l].min(), pilkwang[l].max()),
     "conf_col": l + "__conf", "conf_mean": round(pilkwang[l + "__conf"].mean(), 3),
     "verdict_col": l + "__verdict",
     "YES_%": round(100 * (pilkwang[l + "__verdict"] == "YES").mean(), 1),
     "NO_%": round(100 * (pilkwang[l + "__verdict"] == "NO").mean(), 1),
     "UNK_%": round(100 * (pilkwang[l + "__verdict"] == "UNK").mean(), 1)}
    for l in LABELS
])
display(keys_pilkwang)

report_labels_v2.csv -> 4406 rows x 37 columns
unique StudyInstanceUID: 4406
has a report text column: False


,finding,score_col,score_range,conf_col,conf_mean,verdict_col,YES_%,NO_%,UNK_%
0,ACL,ACL,0.08 - 0.94,ACL__conf,0.813,ACL__verdict,27.8,64.1,8.1
1,MCL,MCL,0.08 - 0.94,MCL__conf,0.789,MCL__verdict,17.4,72.8,9.8
2,Medial Meniscus,Medial Meniscus,0.08 - 0.94,Medial Meniscus__conf,0.858,Medial Meniscus__verdict,53.3,41.1,5.7
3,Lateral Meniscus,Lateral Meniscus,0.08 - 0.94,Lateral Meniscus__conf,0.795,Lateral Meniscus__verdict,25.4,64.5,10.0
4,Medial OA,Medial OA,0.08 - 0.94,Medial OA__conf,0.682,Medial OA__verdict,36.7,37.7,25.5
5,Lateral OA,Lateral OA,0.08 - 0.94,Lateral OA__conf,0.610,Lateral OA__verdict,24.7,42.3,33.1
6,PF OA,PF OA,0.08 - 0.94,PF OA__conf,0.747,PF OA__verdict,45.2,36.3,18.5
7,Effusion,Effusion,0.08 - 0.94,Effusion__conf,0.832,Effusion__verdict,60.5,29.6,9.9
8,Synovitis,Synovitis,0.08 - 0.94,Synovitis__conf,0.189,Synovitis__verdict,13.2,2.6,84.2
9,Baker's,Baker's,0.08 - 0.94,Baker's__conf,0.508,Baker's__verdict,25.3,28.8,45.9


In [6]:
# --- the same 100 studies, in the same order as section 1 ---
pilkwang_idx = pilkwang.set_index("StudyInstanceUID")
pilkwang_sample = pilkwang_idx.reindex(sample_uids)

print("pilkwang rows found for the {} sampled studies: {}".format(
    N_SAMPLE, int(pilkwang_sample.notna().any(axis=1).sum())))
display(pilkwang_sample)

pilkwang rows found for the 100 sampled studies: 100


,ACL,ACL__conf,ACL__verdict,MCL,MCL__conf,MCL__verdict,Medial Meniscus,Medial Meniscus__conf,Medial Meniscus__verdict,Lateral Meniscus,...,Synovitis__verdict,Baker's,Baker's__conf,Baker's__verdict,Contusion,Contusion__conf,Contusion__verdict,Fracture,Fracture__conf,Fracture__verdict
StudyInstanceUID,,,,,,,,,,,,,,,,,,,,,
1.2.826.0.1.3680043.8.498.83376388091077619281568190257019735638,0.08,0.85,NO,0.28,0.05,UNK,0.82,0.95,YES,0.08,...,UNK,0.28,0.05,UNK,0.08,0.85,NO,0.08,0.85,NO
1.2.826.0.1.3680043.8.498.11938864543152001817866013174907274014,0.68,0.95,YES,0.68,0.95,YES,0.82,0.95,YES,0.68,...,UNK,0.28,0.05,UNK,0.82,0.95,YES,0.28,0.05,UNK
1.2.826.0.1.3680043.8.498.49394404966674633152360206835416205330,0.94,0.95,YES,0.08,0.85,NO,0.94,0.95,YES,0.82,...,UNK,0.08,0.85,NO,0.28,0.05,UNK,0.28,0.05,UNK
1.2.826.0.1.3680043.8.498.11402325536521359105285323485008939602,0.94,0.95,YES,0.08,0.85,NO,0.82,0.95,YES,0.08,...,UNK,0.28,0.05,UNK,0.82,0.95,YES,0.28,0.05,UNK
1.2.826.0.1.3680043.8.498.11545541207342029419461179316053148974,0.08,0.85,NO,0.08,0.85,NO,0.08,0.85,NO,0.08,...,UNK,0.28,0.05,UNK,0.08,0.85,NO,0.28,0.05,UNK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1.2.826.0.1.3680043.8.498.19512754570664577075249836633718382506,0.08,0.85,NO,0.08,0.85,NO,0.08,0.85,NO,0.08,...,UNK,0.28,0.05,UNK,0.08,0.85,NO,0.28,0.05,UNK
1.2.826.0.1.3680043.8.498.43445351322695981909807931874796758490,0.08,0.85,NO,0.68,0.95,YES,0.08,0.85,NO,0.08,...,UNK,0.28,0.05,UNK,0.68,0.95,YES,0.08,0.85,NO
1.2.826.0.1.3680043.8.498.91860013313252950044036077977399321593,0.08,0.85,NO,0.08,0.85,NO,0.68,0.95,YES,0.68,...,UNK,0.82,0.95,YES,0.08,0.85,NO,0.08,0.85,NO


## 3 — Both, as rows and columns: are they the same?

Same key, same findings, same studies — what differs is **who filled the labels in, and how many**.

In [7]:
# Both should be displayed as column and rows and are supposed to be same,
# just pilkwang has all reports labeled (verdicts in English) - the reports
# themselves live only in the Kaggle CSV.

uids_kaggle, uids_pilkwang = set(kaggle["StudyInstanceUID"]), set(pilkwang["StudyInstanceUID"])
alignment = pd.DataFrame([
    {"check": "studies (rows)", "kaggle": len(uids_kaggle), "pilkwang": len(uids_pilkwang)},
    {"check": "in both", "kaggle": len(uids_kaggle & uids_pilkwang), "pilkwang": len(uids_kaggle & uids_pilkwang)},
    {"check": "only in this source", "kaggle": len(uids_kaggle - uids_pilkwang), "pilkwang": len(uids_pilkwang - uids_kaggle)},
    {"check": "carries report text", "kaggle": "yes", "pilkwang": "no"},
    {"check": "studies with all 12 labels", "kaggle": int(gold_mask.sum()), "pilkwang": int(pilkwang[LABELS].notna().all(axis=1).sum())},
    {"check": "label type", "kaggle": "binary 0/1 (human)", "pilkwang": "score + conf + verdict (LLM)"},
])
display(alignment)

only_kaggle = sorted(uids_kaggle - uids_pilkwang)
if only_kaggle:
    print("study/ies present in Kaggle but absent from pilkwang:")
    for u in only_kaggle:
        print("  ", u)

,check,kaggle,pilkwang
0,studies (rows),4407,4406
1,in both,4406,4406
2,only in this source,1,0
3,carries report text,yes,no
4,studies with all 12 labels,58,4406
5,label type,binary 0/1 (human),score + conf + verdict (LLM)


study/ies present in Kaggle but absent from pilkwang:
   1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220


In [8]:
# --- one table, the same 100 studies: report on the left, both label sets on the right ---
def side_by_side(uids, findings=("ACL", "Medial Meniscus", "Effusion")):
    """Kaggle report + Kaggle gold label + pilkwang score/verdict, one row per study."""
    kg, pk = kaggle_idx.reindex(uids), pilkwang_idx.reindex(uids)
    out = pd.DataFrame(index=pd.Index(uids, name="StudyInstanceUID"))
    out["report"] = (kg["Report"].astype(str)
                     .str.replace(r"\s+", " ", regex=True).str.slice(0, 70) + "...").values
    out["in_pilkwang"] = pk.notna().any(axis=1).map({True: "yes", False: "no"}).values
    out["kaggle_labelled"] = kg[list(LABELS)].notna().all(axis=1).map({True: "gold", False: "-"}).values
    for f in findings:
        out["kg " + f] = kg[f].values
        out["pk " + f] = pk[f].values
        out["pk " + f + " verdict"] = pk[f + "__verdict"].values
    return out

print("Same {} studies, both sources joined on StudyInstanceUID.".format(N_SAMPLE))
print("kg = Kaggle (blank unless the study is one of the 58 gold ones), pk = pilkwang.\n")
display(side_by_side(sample_uids))

Same 100 studies, both sources joined on StudyInstanceUID.
kg = Kaggle (blank unless the study is one of the 58 gold ones), pk = pilkwang.



,report,in_pilkwang,kaggle_labelled,kg ACL,pk ACL,pk ACL verdict,kg Medial Meniscus,pk Medial Meniscus,pk Medial Meniscus verdict,kg Effusion,pk Effusion,pk Effusion verdict
StudyInstanceUID,,,,,,,,,,,,
1.2.826.0.1.3680043.8.498.83376388091077619281568190257019735638,Regelrechte Stellungsverhältnisse im Kniegelenk. Allseits regelrechtes...,yes,-,NaN,0.08,NO,NaN,0.82,YES,NaN,0.68,YES
1.2.826.0.1.3680043.8.498.11938864543152001817866013174907274014,"SAĞ DİZ MRG. Tetkik protokolü: Çok düzlemli, çok sekanslı. Bulgular: M...",yes,-,NaN,0.68,YES,NaN,0.82,YES,NaN,0.82,YES
1.2.826.0.1.3680043.8.498.49394404966674633152360206835416205330,Prikazane koštane strukture primjerenih intenziteta signala i morfolog...,yes,-,NaN,0.94,YES,NaN,0.94,YES,NaN,0.08,NO
1.2.826.0.1.3680043.8.498.11402325536521359105285323485008939602,ΤΕΧΝΙΚΗ: Η εξέταση έγινε σε Τοµογράφο 3Τ µε ακολουθίες παλµών pd/T2 fs...,yes,-,NaN,0.94,YES,NaN,0.82,YES,NaN,0.82,YES
1.2.826.0.1.3680043.8.498.11545541207342029419461179316053148974,MRI of right knee in sagittal section (Proton density with fat saturat...,yes,-,NaN,0.08,NO,NaN,0.08,NO,NaN,0.28,UNK
...,...,...,...,...,...,...,...,...,...,...,...,...
1.2.826.0.1.3680043.8.498.19512754570664577075249836633718382506,"МР находка: Костите, формиращи дясната колянна става са с нормална фор...",yes,-,NaN,0.08,NO,NaN,0.08,NO,NaN,0.68,YES
1.2.826.0.1.3680043.8.498.43445351322695981909807931874796758490,Diz eklemi içi sıvı miktarı normal. Kuadriseps yağ yastıkcığındaki haf...,yes,-,NaN,0.08,NO,NaN,0.08,NO,NaN,0.08,NO
1.2.826.0.1.3680043.8.498.91860013313252950044036077977399321593,Interprétation : Minime liséré de liquide au sein du récessus supra-pa...,yes,-,NaN,0.08,NO,NaN,0.68,YES,NaN,0.08,NO


### The report language, checked rather than assumed

pilkwang holds no report text, so "all reports in English" cannot be read off it. What *can* be
checked is the language of the Kaggle reports — the thing the labeller had to read. A rough
stop-word / script heuristic (not a language ID model, so treat the split as indicative) says the
corpus is genuinely multi-language, which is exactly why a per-study English verdict is worth having.

In [9]:
# Rough language guess: Greek/Cyrillic by script, the rest by distinctive stop words.
MARKERS = {
    "en": (" the ", " and ", " with ", " without ", " there is ", " normal "),
    "es": (" del ", " con ", " los ", " las ", " se ", " no se "),
    "nl": (" van ", " het ", " een ", " geen ", " niet ", " zijn "),
    "de": (" der ", " die ", " und ", " mit ", " kein ", " nicht "),
    "fr": (" avec ", " une ", " des ", " sans ", " pas de ", " du "),
    "it": (" della ", " con ", " non ", " dei ", " nella "),
    "tr": (" ve ", " bir ", " ile ", " olan ", " izlenmektedir ", " mevcut "),
    "hr/sr": (" je ", " se ", " nema ", " uz ", " bez ", " prikaz "),
}

def guess_language(text):
    if not isinstance(text, str) or not text.strip():
        return "empty"
    if re.search(r"[\u0370-\u03ff]", text):
        return "el"
    if re.search(r"[\u0400-\u04ff]", text):
        return "cyrillic"
    low = " " + text.lower().replace("\n", " ") + " "
    hits = {lang: sum(low.count(m) for m in markers) for lang, markers in MARKERS.items()}
    ranked = sorted(hits.values(), reverse=True)
    best = max(hits, key=hits.get)
    # Needs a few hits and a clear winner, otherwise the shared words decide it.
    return best if ranked[0] >= 3 and ranked[0] > ranked[1] else "other"

kaggle["lang_guess"] = kaggle["Report"].map(guess_language)
lang_mix = (kaggle["lang_guess"].value_counts()
            .rename("studies").to_frame()
            .assign(**{"share_%": lambda d: (100 * d["studies"] / len(kaggle)).round(1)}))
display(lang_mix)

non_english = 100 - lang_mix.loc["en", "share_%"] if "en" in lang_mix.index else 100.0
print("~{:.0f}% of the reports are not English.".format(non_english))
print("pilkwang verdict vocabulary (English, every study):",
      sorted(pilkwang[[l + "__verdict" for l in LABELS]].stack().unique()))

,studies,share_%
lang_guess,,
en,1604,36.4
other,525,11.9
tr,518,11.8
hr/sr,383,8.7
es,379,8.6
el,321,7.3
de,247,5.6
cyrillic,220,5.0
nl,129,2.9


~64% of the reports are not English.
pilkwang verdict vocabulary (English, every study): ['NO', 'UNK', 'YES']


### Where the two agree

The only place the two sources can be compared directly is the **58 gold studies**: human 0/1 on one
side, pilkwang score on the other.

In [10]:
gold_uids = kaggle.loc[gold_mask, "StudyInstanceUID"].tolist()
# Compare only where both sides have a value - one gold study is absent from pilkwang,
# and a missing score must not be scored as a disagreement.
gold_uids = [u for u in gold_uids if u in pilkwang_idx.index]
gold_kg, gold_pk = kaggle_idx.loc[gold_uids], pilkwang_idx.loc[gold_uids]

agreement = pd.DataFrame([
    {"finding": l,
     "gold_positives": int(gold_kg[l].sum()),
     "pk_mean_score_on_positives": round(gold_pk.loc[gold_kg[l] == 1, l].mean(), 3),
     "pk_mean_score_on_negatives": round(gold_pk.loc[gold_kg[l] == 0, l].mean(), 3),
     "agreement_at_0.5": round(((gold_pk[l] >= 0.5).astype(float) == gold_kg[l]).mean(), 3)}
    for l in LABELS
])
display(agreement)
print("compared on {} gold studies present in both sources ({} gold studies in total)".format(
    len(gold_uids), int(gold_mask.sum())))
print("mean agreement at 0.5: {:.3f}".format(agreement["agreement_at_0.5"].mean()))

,finding,gold_positives,pk_mean_score_on_positives,pk_mean_score_on_negatives,agreement_at_0.5
0,ACL,23,0.935,0.279,0.842
1,MCL,9,0.847,0.226,0.842
2,Medial Meniscus,26,0.832,0.242,0.860
3,Lateral Meniscus,23,0.751,0.320,0.772
4,Medial OA,15,0.799,0.274,0.842
5,Lateral OA,11,0.693,0.290,0.842
6,PF OA,20,0.729,0.269,0.789
7,Effusion,34,0.784,0.527,0.719
8,Synovitis,27,0.504,0.325,0.702
9,Baker's,12,0.750,0.277,0.895


compared on 57 gold studies present in both sources (58 gold studies in total)
mean agreement at 0.5: 0.795


### Re-roll the sample

`show(n, seed)` draws a different set of studies and re-renders the joined view.

In [11]:
def show(n=N_SAMPLE, seed=SEED, findings=("ACL", "Medial Meniscus", "Effusion")):
    """Draw n random studies and render the Kaggle + pilkwang view for them."""
    uids = kaggle["StudyInstanceUID"].sample(n=n, random_state=seed).tolist()
    return side_by_side(uids, findings=findings)

show(n=10, seed=7)

,report,in_pilkwang,kaggle_labelled,kg ACL,pk ACL,pk ACL verdict,kg Medial Meniscus,pk Medial Meniscus,pk Medial Meniscus verdict,kg Effusion,pk Effusion,pk Effusion verdict
StudyInstanceUID,,,,,,,,,,,,
1.2.826.0.1.3680043.8.498.39578126940710463296572383286587298912,Diz eklemi içi sıvı miktarı normal. Çapraz ve yan bağlar normal. Medya...,yes,-,NaN,0.08,NO,NaN,0.08,NO,NaN,0.08,NO
1.2.826.0.1.3680043.8.498.81590494282169434250974061642267472166,Antecedentes Clínicos: Esguince del ligamento colateral lateral de rod...,yes,-,NaN,0.08,NO,NaN,0.08,NO,NaN,0.08,NO
1.2.826.0.1.3680043.8.498.69596668032633976121623939445795750384,"SAĞ DİZ MRG. Tetkik protokolü: Çok düzlemli, çok sekanslı. Bulgular: M...",yes,-,NaN,0.08,NO,NaN,0.82,YES,NaN,0.82,YES
1.2.826.0.1.3680043.8.498.60094182309575937331152278560900767484,Técnica: RMN de la rodilla. Resultados: Condropatía rotuliana. Derrame...,yes,-,NaN,0.28,UNK,NaN,0.28,UNK,NaN,0.82,YES
1.2.826.0.1.3680043.8.498.43339007024446439709455570527734115557,"SAĞ DİZ MRG. Tetkik protokolü: Çok düzlemli, çok sekanslı. Bulgular: L...",yes,-,NaN,0.68,YES,NaN,0.82,YES,NaN,0.68,YES
1.2.826.0.1.3680043.8.498.38717329577625255474350472405748992219,Technique: MRI of the knee. ACL normal. MCL normal. Degenerative ruptu...,yes,-,NaN,0.08,NO,NaN,0.82,YES,NaN,0.08,NO
1.2.826.0.1.3680043.8.498.10432568348356281020515782178064869805,"МР находка: МР данни за ставен излив с белези на хемартроза. Костите, ...",yes,-,NaN,0.94,YES,NaN,0.08,NO,NaN,0.82,YES
1.2.826.0.1.3680043.8.498.59511171909586924989766685687032462369,CONSTATATIONS : Fractures : Aucune. Alignement articulaire : Normal. C...,yes,-,NaN,0.08,NO,NaN,0.08,NO,NaN,0.08,NO
1.2.826.0.1.3680043.8.498.66305771884507759587182077473863996177,Antecedentes Clínicos: Artrosis patelofemoral. Obs. Lesión meniscal la...,yes,-,NaN,0.08,NO,NaN,0.68,YES,NaN,0.68,YES


## Takeaways

- **Same keys, same studies.** Both tables are one row per `StudyInstanceUID`, 4,407 on the Kaggle
  side and 4,406 on the pilkwang side — one study is missing from pilkwang and is named in section 3.
- **Kaggle is nearly unlabelled.** 58 studies out of 4,407 carry the 12 labels; the other 4,349 are a
  report and nothing else. That is the gap pilkwang fills.
- **pilkwang has no report text.** It is labels only — score, confidence and an English
  `YES` / `NO` / `UNK` verdict per finding, derived from the original-language report.
- **`UNK` is information, not a gap.** It marks findings the report never mentions — 84% for
  Synovitis, 56% for Fracture — so a low score there means *unsaid*, not *absent*.
- **The two line up where they can be compared.** On the 57 gold studies pilkwang also covers, its
  mean score separates the classes on every finding - 0.50-0.94 on true positives against 0.23-0.53
  on true negatives, 0.795 mean agreement at a 0.5 threshold. Synovitis (0.50 vs 0.33) and Contusion
  (0.75 vs 0.48) are the narrowest gaps, and both are findings reports often leave unsaid.